# One check, taken apart

The reconciliation loop closes the difference between what should exist and
what does. Everything it ever does is one **check**, three steps long:

| step | question | reads | writes |
|---|---|---|---|
| **observe** | is the thing intent implies actually in storage? | S3, at the addresses intent implies | `materialized_models`, `materialized_nd_runs` |
| **gap** | what should exist, minus what does | a snapshot | nothing — it is a pure function |
| **act** | close the difference | — | submits a job, or records why not |

Two ideas carry the whole design, and this notebook shows each one happening:

**Intent implies an address.** A model's identity is a hash of the inputs that
define it — methodology commit, reach geometry, DEM source, and so on. All of
those live in the database, so the loop can compute where a model *must* be
before any job runs. Observation is a lookup, never a search.

**Results flow upstream, so work flows downstream-first.** A reach's model
needs its downstream neighbour's model *and* ND library (the max-q stage
transfer line shapes this reach's geometry). Terminal reaches — outlets — have
no downstream, so a fresh network starts building at its outlets and everything
else waits its turn.

**Before running:** `docker compose up -d db minio minio-init`, and both the
`build_model` and `run_nd_scenarios` images available locally. Section 9 runs a
real hydraulic simulation per discharge, so budget minutes to hours for it —
narrow the reach's discharge range in `desired_state` first if you only want to
see the mechanism.

In [ ]:
import json
import time

import pandas as pd

from recon import activity, check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.config import settings
from recon.workers import LocalDockerRunner

pd.set_option("display.max_colwidth", 42)

print(f"database  {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print(f"storage   s3://{settings.artifacts_s3_bucket} via {settings.aws_endpoint_url}")
print(f"images    {settings.build_model_image}\n          {settings.run_nd_scenarios_image}")

## 1. Author intent

Nothing exists yet, and nothing is asked for. `scripts/seed.py` changes that,
and it is the same script a deployment would run:

- **`desired_state_defaults`** — one row, the deployment's intent: the identity
  inputs (methodology commit, grid resolution, EPSG, DEM and LULC sources and
  lookup), the solver and its version, and fallbacks for everything else. These
  must match what the deployed images use, because the loop predicts addresses
  from them. `sdr_commit` and the solver version are baked into the images and
  cannot be overridden — the row records what the image is expected to report,
  and a disagreement shows up as artifacts never found where the loop looked.
- **`reach_network`** — the topology, from the hydrofabric.
- **`lakes`** — the polygons terminal reaches drain into, exported to storage
  as GeoJSON so the ND job can be handed a path to one.
- **`desired_state`** — one row per reach, all fields NULL: *"I want this
  reach, defaults for everything."* A reach in the network means nothing until
  this row exists.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
import seed

seed.seed(seed.DEFAULT_NETWORK_GPKG, seed.DEFAULT_NHF_GPKG)

# Start from nothing in storage, so every address in this notebook is one the
# loop predicts and then fills.
s3 = storage.get_s3_client()
for prefix in ("version=v1/models/", "version=v1/results/"):
    for page in s3.get_paginator("list_objects_v2").paginate(
            Bucket=settings.artifacts_s3_bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            s3.delete_object(Bucket=settings.artifacts_s3_bucket, Key=obj["Key"])
with db.connect() as conn:
    conn.execute("TRUNCATE materialized_models, materialized_nd_runs, "
                 "materialized_kwse_runs, reach_processing, reach_activity")

display(pd.DataFrame(db.table_counts()))

## 2. The queue is a question

There is no queue data structure. Asking the database *which reaches need
looking at* **is** the queue — which is why a reconciler that dies mid-sweep
loses nothing. Each row says why it is due.

In [ ]:
due = pd.DataFrame(queue.due_reaches())
print(f"{len(due)} reaches due")
due.head(6)

## 3. Effective intent, and the address it implies

Pick two reaches: a **terminal** (an outlet — no downstream) and a
**non-terminal** just above one. For each, the loop resolves effective intent
(`COALESCE(desired_state.x, desired_state_defaults.x)`), builds the identity
object the job would build, and hashes it.

That hash is the address. No job has run, yet we know exactly where each
reach's model must appear in the bucket.

In [ ]:
rows = db.query("""
    SELECT rn.reach_id, rn.is_terminal FROM reach_network rn
    JOIN reach_network ds ON ds.reach_id = rn.reach_to_id
    WHERE ds.is_terminal LIMIT 1""")
upstream_id = rows[0]["reach_id"]
terminal_id = db.one("SELECT reach_to_id AS r FROM reach_network WHERE reach_id=%s", (upstream_id,))["r"]
print(f"terminal reach:     {terminal_id}")
print(f"non-terminal above: {upstream_id}\n")

wanted = intent.effective(terminal_id)
identity_obj, identity_hash = identity.model_identity(wanted)
print("identity object the job will build:")
print(json.dumps(identity_obj, indent=2))
print(f"\npredicted identity hash: {identity_hash}")
print(f"predicted address:       {storage.model_base_path(terminal_id)}/{identity_hash}_<domain>/")

## 4. Observe — a lookup, not a search

Observe looks at that one address. Anything else in the bucket — models from
older intent, a neighbour's artifacts — is invisible to it, so there is
nothing to rank and no "newest wins".

Nothing is there yet, so no proof row is written.

In [ ]:
print("observe:", observe.observe_reach(terminal_id))
print("proof rows:", db.query("SELECT * FROM materialized_models"))

## 5. Gap — a pure function decides

The snapshot is one query: this reach's proofs, its downstream neighbour's
proofs, and whether a job is already in flight. `gap.calculate` is pure — no
database, no storage, no clock — so the same snapshot always gives the same
answer, and you can read every rule the loop has in one file.

Watch the ladder treat the two reaches differently: the terminal may build;
the non-terminal must wait, and the decision says for whom.

In [ ]:
for rid, label in ((terminal_id, "terminal"), (upstream_id, "non-terminal")):
    snap = check.load_snapshot(rid)
    print(f"{label} {rid}:")
    print(f"   snapshot: model_ok={snap.model_ok} ds_model_ok={snap.ds_model_ok} ds_nd_ok={snap.ds_nd_ok}")
    print(f"   decision: {gap.calculate(snap)}\n")

## 6. Act — submit and walk away

`run_check` performs all three steps and acts on the decision. For the
terminal that means submitting a real `build_model` container — and returning
immediately. The fact that work is running lives in the **database**
(`current_step`, with the container id as the handle), not in this notebook's
memory: kill the kernel now and nothing is lost.

Checking again while the job runs does not resubmit — that is the in-flight
marker doing its job. Checking the non-terminal records who it waits for.

In [ ]:
from recon.workers import job_env

# job_env() carries the S3 configuration the images need. Worth knowing why it
# exists: TWO S3 clients live inside them and they are configured separately.
# boto3 reads AWS_ENDPOINT_URL and handles artifact copying; GDAL ignores it
# entirely, and run_nd_scenarios loads the inflow line, centerline and outflow
# polygon through geopandas -> pyogrio -> GDAL. Without GDAL's own variables
# those reads resolve against real AWS and come back 403 — an authentication
# error whose actual cause is the endpoint.
#
# USE_CUDA is emptied because this machine has no GPU. The image is built with
# CUDA support but does not require it: the flag only adds a `cuda` line to the
# solver's parameter file. Drop it where a GPU is available.
#
# No image argument: the runner picks one per step, because the job name is
# baked into each image's ENTRYPOINT.
runner = LocalDockerRunner(
    network=settings.docker_network,
    env_vars=job_env({"USE_CUDA": ""}),
    platform=settings.docker_platform,
    volumes=[f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else [])

print("terminal:    ", check.run_check(terminal_id, runner))
print("check again: ", check.run_check(terminal_id, runner), "   <- no resubmit")
print("non-terminal:", check.run_check(upstream_id, runner))
print()
print(pd.DataFrame(processing.in_flight()))

## 7. Hear back

The **job status pass** asks the execution system what became of the jobs the
database says are in flight. It records nothing about what exists — it only
clears the marker and requests a check. Whether anything was *produced* is
storage's question, answered by the next observe.

In [ ]:
deadline = time.time() + 900
while time.time() < deadline:
    outcomes = jobs.status_pass(runner)
    if not outcomes:
        print("nothing in flight")
        break
    print(f"{time.strftime('%H:%M:%S')}  {outcomes[0]['status']:<10} {outcomes[0]['action']}")
    if outcomes[0]["status"] in ("succeeded", "failed"):
        break
    time.sleep(15)

## 8. The next check finds the proof

Observe looks at the predicted address again — and now the manifest is there.
Before adopting it, the loop verifies it: the manifest must belong to this
reach, sit in the folder its hash names, carry exactly the identity fields the
loop knows, and its identity object must re-hash to the value it claims. Only
then is the proof row written, stamped with the revision it proves.

The number to check: the job computed its identity **independently, inside the
container** — and landed on the hash we predicted in step 3.

In [ ]:
print(check.run_check(terminal_id, runner))
print()
row = db.one("SELECT * FROM materialized_models WHERE reach_id=%s", (terminal_id,))
print("proof row: ", row)
print(f"\npredicted {identity_hash} == adopted {row['identity_hash']}:", identity_hash == row["identity_hash"])

## 9. The next rung — the normal-depth library

The terminal's model is proved, so the same loop moves up one rung. `run_nd_scenarios`
sweeps a range of discharges and writes one scenario per hydraulically distinct
step; the loop hands it the discharge range from intent and the polygon of the
lake this reach drains into.

Note what the loop can and cannot predict here. The library's address is fixed
by intent down to the slope — `results/reach=<id>/<model identity>/<run identity>/nd=<slope>/`
— so getting there is a lookup. **Which discharges are inside is not intent's to
say:** the job's adaptive step algorithm decides, and the loop reads them back.
Only the two ends are guaranteed, and that is exactly what it checks — the
library must span what intent asked for.

This is a real hydraulic simulation per discharge, so it takes far longer than
the build. The cell below submits and then waits.

In [ ]:
print("terminal:", check.run_check(terminal_id, runner), "   <- model proved, nd is the gap now")

deadline = time.time() + 7200
while time.time() < deadline:
    outcomes = jobs.status_pass(runner)
    if not outcomes:
        break
    print(f"{time.strftime('%H:%M:%S')}  {outcomes[0]['status']:<10} {outcomes[0]['action']}")
    if outcomes[0]["status"] in ("succeeded", "failed"):
        break
    time.sleep(30)

print("\nafter the run:", check.run_check(terminal_id, runner))
row = db.one("SELECT * FROM materialized_nd_runs WHERE reach_id=%s", (terminal_id,))
print("\nproof row:")
for key in ("run_identity_hash", "q_set", "us_wse_max", "applied_revision"):
    print(f"   {key:20} {row[key]}")
print(f"   {'us_min_wse_curve':20} {row['us_min_wse_curve']}")

## 10. The wave moves upstream

The non-terminal has been waiting all along — not for a model, but for the
downstream **library**, because its own geometry uses the downstream max-q
stage transfer line. That library now exists, so the same check that returned
`WaitingDownstream` before returns `RunStep` now. Nothing was told; the fact
appeared in the database and the next check read it.

This is the whole cascade in miniature. Run `03_run_network.ipynb` to watch it
travel the network.

In [ ]:
snap = check.load_snapshot(upstream_id)
print(f"snapshot: model_ok={snap.model_ok} ds_model_ok={snap.ds_model_ok} ds_nd_ok={snap.ds_nd_ok}")
print("decision:", gap.calculate(snap))
print()
print("non-terminal now:", check.run_check(upstream_id, runner))
print()
display(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))
display(pd.DataFrame(activity.recent(8, reach_id=terminal_id))[
    ["action", "outcome", "revision", "detail"]])

## What to remember

- **A check is short and never waits.** Three checks saw one job through:
  submit, leave alone, adopt. A crash between any two costs nothing.
- **Intent implies the address.** The job and the loop computed the same hash
  from the same inputs without talking to each other — that is what lets
  observation be a single lookup, and what makes two models in one bucket
  unambiguous.
- **The row is proof.** `materialized_models` has a row exactly when a reach's
  model intent is satisfied, stamped with the revision it proves. Deleting the
  model from storage deletes the row, and the claim with it.
- **Dependencies gate work, not proof.** The ladder decides what may *start*;
  what already exists at the right address is adopted no matter what.
- **Prediction stops where authorship stops.** Intent fixes a library's address
  down to the slope, so the loop looks there directly; the discharges inside are
  the job's to choose, so the loop reads them back and judges whether they span
  what was asked. Everything the loop cannot author, it accepts as found.

`02_cases.ipynb` runs the awkward situations — deletions, tampering, intent
changes, failures. `03_run_network.ipynb` lets the loop run the whole network.